# Set Variables

In [ ]:
# -------- bad smell
# bad_smell = "CC" # Cyclomatic Complexity
bad_smell = "DE" # Design
# bad_smell = "NC" # Npath Complexity


# --------
my_keyword_Based = True
my_conv_kernel_output_channel = 64
my_balance_train = False


# --------
if my_conv_kernel_output_channel == 64:
    output_version = 2
else: # my_conv_kernel_output_channel == 8
    output_version = 3


if my_balance_train:
    file_subversion = ".2-balanced"
else:
    file_subversion = ".1-imbalance"

# Google Colab

In [ ]:
sub_folders = {
    "camel":      "camel_"       + bad_smell,
    "cloudstack": "cloudstack_"  + bad_smell,
    "geode":      "geode_"       + bad_smell,
    "hbase":      "hbase_"       + bad_smell
}

In [ ]:
dataset_file_names = {
    "camel":      "camel_DE - v.02",
    "cloudstack": "cloudstack_DE - v.01",
    "geode":      "geode_DE - v.01",
    "hbase":      "hbase_DE - v.01"
}

In [ ]:
tempPre = "E:/darsy/00/02- arshad/10- paper code/"
tempOutput = "01- Jupyter Notebook/Direct Method/00. Output/"

In [ ]:
pre_path_outputs = {}
for k, v in dataset_file_names.items():
    pre_path_outputs[k] = tempPre + tempOutput + k + "/" + sub_folders[k] + "/" + v + "/"

# Libs

In [ ]:
import json
import os.path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from operator import truediv

from matplotlib.ticker import MaxNLocator

In [ ]:
# import nltk
# nltk.download("stopwords")

In [ ]:
!python -V
import matplotlib
print(matplotlib.__version__)

# Config

In [ ]:
mypaths = {}

for k, v in dataset_file_names.items():
    if my_keyword_Based:
        mypaths[k] = pre_path_outputs[k] + "performances/tfidf-performance-v{}{}.json".format(output_version, "{}")
    else:
        mypaths[k] = pre_path_outputs[k] + "performances/w2v-performance-v{}{}.json".format(output_version, "{}")
    
    # -----
    
    if my_balance_train:
        mypaths[k] = mypaths[k].format(".2-balanced")
    else:
        mypaths[k] = mypaths[k].format(".1-imbalance")

In [ ]:
mypaths

In [ ]:
preprocessing_params = {
    "keyword_Based": my_keyword_Based
}

In [ ]:
bcd_colours = ["blue", "green", "red"]

# my_whole_datas

In [ ]:
my_whole_datas = {}
for k, v in mypaths.items():
    with open(v) as filehandle:
        my_whole_datas[k] = json.load(filehandle)

In [ ]:
class_labels = my_whole_datas["camel"]["preprocessing_params"]["bug_classes"]
epochs = my_whole_datas["camel"]["train_params"]["epochs"]

In [ ]:
print(class_labels)
print(epochs)

In [ ]:
best_step_labels = {}
best_step_indexs = {}
datas = {}

for k, mwd in my_whole_datas.items():
    best_step_labels[k] = mwd["model_results"]["validation"]["best_at_step"]
    best_step_indexs[k] = mwd["model_results"]["validation"]["best_at_step"] - 1
    datas[k] = mwd["model_results"]

In [ ]:
print(best_step_labels)
print(best_step_indexs)

In [ ]:
ls = "solid"
lw = 1
lw_test = 5
marker = "."
marker_test = "o"
# train_data_status = train_data_status

colours = {
    "camel":      "red",
    "cloudstack": "blue",
    "geode":      "green",
    "hbase":      "orange"
}

# plot_acc_cost_

In [ ]:
for k, mwd in my_whole_datas.items():
    print("-" * 10, k)
    for i, acc in enumerate(datas[k]["validation"]["accuracy"]):
        print(i + 1, round(acc, 1))

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 10))

mymetrics = ["accuracy", "cost"]
my_x = range(1, epochs + 1, 1)

for ax, metric in zip(axes, mymetrics):
    ax.get_xaxis().set_major_locator(MaxNLocator(integer=True))

    if metric == "accuracy":
        for k, mwd in my_whole_datas.items():
            ax.set_ylim([0, 100])
            ax.scatter(best_step_labels[k], datas[k]["validation"]["best_acc"], 
                       label="best on Validation Dataset", color="cyan", lw=lw_test, marker=marker_test)
    else:
        for k, mwd in my_whole_datas.items():
            ax.set_ylim([0, 300])
            ax.scatter(best_step_labels[k], datas[k]["validation"]["cost"][best_step_indexs[k]], 
                       label="best on Validation Dataset", color="cyan", lw=lw_test, marker=marker_test)

    for k, mwd in my_whole_datas.items():
        ax.plot(my_x, metric, data=datas[k]["validation"], label="Validation Dataset - {}".format(k), 
                color=colours[k], linestyle=ls, lw=lw, marker=marker)

    ax.set_xlim([0, 30])
    ax.set_xlabel("epoch")
    ax.set_ylabel(metric)
    ax.legend()
    ax.grid(True)

    ax.get_xaxis().set_ticks( list(range(1, epochs + 1)) )
# --- end for

fig.tight_layout(pad=1.0)

# call

In [ ]:
my_path =   "D:/SAVE/Tests/"
file_name = "Direct Method - v06-dpi=1000.jpg"
full_path = my_path + file_name

resolution_value = 1000

fig.savefig(full_path, format="jpg", dpi=resolution_value)